# Run experiments on Colab (GPU or TPU)

**Before running:** Menu → *Runtime* → *Change runtime type* → set **Hardware accelerator** to **GPU** (T4) or **TPU** (e.g. v5e-1) → Save.

Then run the cells below in order.

In [ ]:
# Check GPU or TPU
import os

try:
    import torch
    has_cuda = torch.cuda.is_available()
    has_tpu = bool(os.environ.get("COLAB_TPU_ADDR"))
    if has_cuda:
        print("GPU:", torch.cuda.get_device_name(0))
    elif has_tpu:
        print("TPU detected (e.g. v5e-1). Run the next cell to install torch_xla, then run the experiment.")
    else:
        print("No GPU/TPU. Go to Runtime → Change runtime type → set GPU or TPU and re-run.")
except Exception as e:
    print("Error:", e)

In [ ]:
# Install dependencies; if using TPU, install PyTorch/XLA
import os

!pip install -q transformers datasets accelerate scikit-learn pyyaml tqdm
if os.environ.get("COLAB_TPU_ADDR"):
    !pip install -q torch_xla
    print("torch_xla installed for TPU.")

In [ ]:
# Optional: use Hugging Face token (avoids rate limits; use Colab Secrets, don't paste token in notebook)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set from Colab Secrets.")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        print("No HF_TOKEN in Secrets. Add it: key icon (Secrets) → Add new secret → name: HF_TOKEN, value: your token.")

## Option A: Your project is in a GitHub repo

Set `REPO_URL` below to your repo (e.g. `https://github.com/yourname/your-repo.git`) and run the next cell.

In [ ]:
REPO_URL = "https://github.com/code-ninja-bayyavarapu/Role-Aware-and-Confidentiality-Constrained-LLM.git"

if REPO_URL:
    !git clone --depth 1 "$REPO_URL" repo
    %cd repo
    PROJECT_ROOT = "."
    print("Cloned. Project root:", PROJECT_ROOT)
else:
    PROJECT_ROOT = "."
    print("No REPO_URL set. Use Option B to upload a zip of your project.")

## Option B: Upload project zip

If you don't use GitHub: zip your project folder (the one that contains `main.tex` and `experiments/`), then run the next cell and upload the zip when prompted.

In [ ]:
from google.colab import files
import zipfile
import os

if not os.path.isdir("repo") or not os.path.isfile("repo/experiments/run_experiments.py"):
    uploaded = files.upload()  # Opens file picker
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name, "r") as z:
                z.extractall(".")
            PROJECT_ROOT = "."
            for root, dirs, fnames in os.walk("."):
                if "run_experiments.py" in fnames and "experiments" in root:
                    PROJECT_ROOT = os.path.dirname(root)  # parent of experiments/
                    break
            print("Extracted. Project root:", os.path.abspath(PROJECT_ROOT))
            break
    else:
        print("Upload a .zip of your project.")
else:
    PROJECT_ROOT = "repo"
    print("Using cloned repo.")

os.chdir(PROJECT_ROOT)

In [ ]:
# Run experiments (quick mode, uses GPU if available)
import os
os.chdir(PROJECT_ROOT)
!python experiments/run_experiments.py --config experiments/config.yaml --quick

In [ ]:
# Zip results and download
from google.colab import files
import zipfile
import os

out_dir = "experiments/outputs"
if os.path.isdir(out_dir):
    zip_name = "experiment_results.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for r, ds, fs in os.walk(out_dir):
            for f in fs:
                path = os.path.join(r, f)
                z.write(path, path)
    files.download(zip_name)
    print("Downloaded", zip_name)
else:
    print("No outputs folder found. Run the experiment cell first.")